In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

from IPython.display import display, HTML

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Key take-away:

`Keywords extraction` is the **NLP** technique that involves identifying and extracting the most important **words** or **phrases** from a piece of text. The extracted keywords can be used 

* to summarize the content of the text;
* to identify the main topics and themes discussed in the text, or
* to facilitate information retrieval.

`YakeKeywordExtraction` is a **keyword extraction** technique implemented in Spark NLP. Yake is a novel feature-based system for **multi-lingual** extracting keywords from a text, which supports texts of different sizes, domains or languages.

### To install Spark NLP and extract keywords in Python, simply use your favorite package manager (conda, pip, etc.).

In [2]:
!pip install spark-nlp pyspark nltk spark-nlp-display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.8/758.8 kB 14.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.9/66.9 kB 3.7 MB/s eta 0:00:00


### Then, simply import the library and start a Spark session:

In [3]:
import sparknlp
#import sparknlp_jsl

# Import Spark NLP
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline
from sparknlp_display import NerVisualizer

import pyspark.sql.functions as F
from pyspark.sql.functions import *

import warnings

In [4]:
# Settings and parameters for the Spark session
# As included in JSL notebooks

warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"16G", 
          "spark.kryoserializer.buffer.max":"2000M", 
          "spark.driver.maxResultSize":"2000M"} 

print("Spark NLP Version :", sparknlp.version())

# Staring Healthcare Spark NLP session
spark = sparknlp.start(params=params)
spark

Spark NLP Version : 6.3.3
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f778c3db-726e-48d3-b067-aaa0502df285;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.g

In [5]:
! wget -q https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/pubmed/pubmed_sample_text_small.csv

The `YakeKeywordExtraction` annotator expects **TOKEN** as input, and then will provide **CHUNK** as output. Thus, we need the previous steps to generate those annotations that will be used as input to our annotator.

In [6]:
# Step 1: Transforms raw texts to `document` annotation
document = DocumentAssembler() \
.setInputCol("text") \
.setOutputCol("document")

In [7]:
# Step 2: Sentence Detection
sentenceDetector = SentenceDetector() \
.setInputCols("document") \
.setOutputCol("sentence")

In [8]:
# Step 3: Tokenization
token = Tokenizer() \
.setInputCols("sentence") \
.setOutputCol("token") \
.setContextChars(["(", ")", "?", "!", ".", ","])

In [9]:
# Step 4: Keyword Extraction
keywords = YakeKeywordExtraction() \
.setInputCols("token") \
.setOutputCol("keywords") \
#.setThreshold(0.6f) \
#.setMinNGrams(2) \
#.setNKeywords(10)

In [10]:
# Define the pipeline
yake_pipeline = Pipeline(
    stages=[
        document, 
        sentenceDetector, 
        token, 
        keywords
    ])

In [11]:
# Create an empty dataframe
empty_df = spark.createDataFrame([['']]).toDF("text")

In [12]:
# Fit the dataframe to get the 
yake_Model = yake_pipeline.fit(empty_df)

In [13]:
sample_text = """I was in JGH emergency with acute atrial fibrillation attack, during the whole day JGH emergency did just ECG confirming atrial fibrillation and that was all. 
No doctor appear in the room to see or treat me. At 3am I was told by a nurse to go home untreated. I was given professional advice and treatment as well. 
Called me in am & pm for first 3 days following my hip surgery also when called after 15 days, they were very professional and with a human respect. 
In the ICU the care and dedication of the staff was incredible. The doctor went above and beyond what I would think as normal care to help me through this life-threatening experience. 
Super quick. I brought in my dad thinking one thing but triage nurse realized the issue right away. And we got treated/looked at in a timely manner. 
The physiotherapist was excellent. He also arranged with the nurse practitioner to recommend a X-ray which ultimately led to a diagnosis of bone metastates of breast cancer. 
excellent attention. Dr and technician were considerate and explained everything clearly. Very organized, registered at the kiosk but too bad the screen where the numbers were displayed was not functioning that day. 
This system works very well. The protocol has been well followed. Taken on time for my appointment. Technician was very competent, friendly and put me at ease.	
Fast, professionel, and competant. I don't think I could get better care anywhere else. Ambulance ride was very well done and I was taken care of quickly and efficiently. 
My primary nurse who was young was great and empathetic. I was committing blood and didn't receive any care waiting for more than 6 hours and wasn't able to stay in the emergency and go back home. 
Very caring and personal. Very supportive. Nurse Leora was very good - shot was painless. The wait was not too long and the personnel were nice and helpfull. 
They ran all the necessary test to eliminate the possible causes. In the future if I need to go to an emergency room, I will go to the Jewish General. 
The staff and the hospital are excellent (10 out of 10). It's hard to say my experience was excellent because cancer is a hard thing to go through and mostly an unpleasant experience. 
That being said, I cannot say enough about how excellent the staff and the care is at the Jewish General Hospital. i was listen, oriented and diagnose. Caring emerg doctors and specialists.
I was treated immediately and great doctor and staff. When I was able to be seen by the physician, they evaluated me and then proceeded with tests. 
All personnel and especially the doctors took great care of me and helped me mentally get through this. A good oncology surgeon,plastic surgeon, nurses and a whole team were very good.
fast courteous. Kind, Respectful, Knowledgeable care. Informed re next steps. Doctors were excellent and answered questions. Follow-up arranged. I can walk and be independent.
The Doctor was very good and so we're the nurse. All appropriate testing was done. Fast, efficient care. Took very good care of me. Ran all tests needed and got results, kept me overnight. 
Went home with instructions and meds. Feeling much better now. Fast, efficient care. Victor came to Check my house to make sure it was safe for me. Very nice, kind and understanding doctors.
They saved my life. All the Staff at Maimonides offer exceptional care for my mother who is bedridden and cannot walk at all. I arrived at the ER with a heart rate of 177 and I was hooked up within minutes of registering. 
Also, my wait to register was under 5 minutes. As well, the doctors, nurses and PABs were all attentive and respectful. The staff were so courteous and gentle explaining things well with the different steps."""

In [14]:
# Split into sentences using regex
sentences = re.split(r'(?<=[.!?])\s+', sample_text.strip())

In [15]:
# Create DataFrame
df = pd.DataFrame(sentences, columns=['text'])

In [16]:
# Show first 10 rows
print(df.head(10))

                                                                                                                                                             text
0  I was in JGH emergency with acute atrial fibrillation attack, during the whole day JGH emergency did just ECG confirming atrial fibrillation and that was all.
1                                                                                                                No doctor appear in the room to see or treat me.
2                                                                                                              At 3am I was told by a nurse to go home untreated.
3                                                                                                          I was given professional advice and treatment as well.
4            Called me in am & pm for first 3 days following my hip surgery also when called after 15 days, they were very professional and with a human respect.
5                           

In [17]:
df.to_csv('jgh_sample_text_small.csv', index=False)

In [18]:
spark_df = spark.read\
                .option("header", "true")\
                .csv("jgh_sample_text_small.csv")\
                
spark_df.show(truncate=300)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                                                          text|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------+
|I was in JGH emergency with acute atrial fibrillation attack, during the whole day JGH emergency did just ECG confirming atrial fibrillation and that was all.|
|                                                                                                              No doctor appear in the room to see or treat me.|
|                                                                                                            At 3am I was told by a nurse to go home untreated.|
|                                 

In [19]:
lmodel = LightPipeline(yake_Model)

In [20]:
lresult = lmodel.fullAnnotate(sample_text)[0]

In [21]:
keys_df = pd.DataFrame([(k.result, k.begin, k.end, k.metadata['score'],  k.metadata['sentence']) for k in lresult['keywords']],
                       columns = ['keywords','begin','end','score','sentence'])
keys_df['score'] = keys_df['score'].astype(float)

# ordered by relevance 
keys_df.sort_values(['score']).head(1000)

,keywords,begin,end,score,sentence
0,jgh emergency,9,21,0.286330,0
5,jgh emergency,83,95,0.286330,0
21,jewish general,2310,2323,0.506522,32
20,jewish general,2018,2031,0.506522,29
4,day jgh,79,85,0.633203,0
6,ecg confirming,106,119,0.774866,0
2,atrial fibrillation,34,52,0.864581,0
8,atrial fibrillation,121,139,0.864581,0
26,efficient care,3009,3022,1.085182,47
28,efficient care,3179,3192,1.085182,52


In [22]:
# Fit and transform the dataframe to get the predictions
result = yake_pipeline.fit(spark_df).transform(spark_df)

In [23]:
print(result)

DataFrame[text: string, document: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, sentence: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, token: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, keywords: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>]


In [24]:
result = result.withColumn('unique_keywords', F.array_distinct("keywords.result"))

In [25]:
result.show(1, truncate=300)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------

In [26]:
def highlight(text, keywords):
    for k in keywords:
        text = (re.sub(r'(\b%s\b)'%k, r'<span style="background-color: #90EE90;">\1</span>', text, flags=re.IGNORECASE))
    return text

In [27]:
highlight_udf = udf(highlight, StringType())

In [28]:
result = result.withColumn("highlighted_keywords",highlight_udf('text','unique_keywords'))

In [29]:
for r in result.select("highlighted_keywords").limit(20).collect():
    display(HTML(r.highlighted_keywords))
    print("\n\n")